# Script to calculate Cultural diversity based on CUBE

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from vendi_score import vendi  
import re
import traceback

def normalize(name):
    return name.lower().replace(' ', '-').replace('_', '-')


def calculate_cultural_diversity(
    quality_scores_csv,
    tagged_artifacts_folder,
    w1=1.0,
    w2=0.0,
    w3=0.0,
    batch_size=8,
    return_components=False
):
    
    quality_df = pd.read_csv(quality_scores_csv)
    quality_dict = {}
    
    
    for _, row in quality_df.iterrows():
        prompt_name = str(row['prompt name']).strip()
        result_str = str(row['Result']).strip()
        
        try:
            quality_score = float(result_str)
            
            quality_dict[prompt_name] = quality_score
            quality_dict[normalize(prompt_name)] = quality_score
            
        except Exception as e:
            print(f"Warning: Could not parse quality score for prompt '{prompt_name}': {result_str} - {e}")
            continue
    
    
    folder_path = Path(tagged_artifacts_folder)
    
    if not folder_path.exists():
        raise ValueError(f"Folder not found: {folder_path}")
    
    all_annotations = []
    csv_files = list(folder_path.glob('*.csv'))

    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)

            filename_prompt = csv_file.stem
            quality_score = None

            if filename_prompt in quality_dict:
                quality_score = quality_dict[filename_prompt]
            

            elif normalize(filename_prompt) in quality_dict:
                quality_score = quality_dict[normalize(filename_prompt)]
            
            else:
                for original_prompt in quality_df['prompt name']:
                    if normalize(original_prompt) == normalize(filename_prompt):
                        quality_score = quality_dict[normalize(original_prompt)]
                        break
            
            if quality_score is None:
                print(f"No quality score found for file '{csv_file.name}'")
                continue

            for _, row in df.iterrows():
                annotation = {
                    'image_name': row['image_name'],
                    'continent': str(row['continent']).strip(),
                    'country': str(row['country']).strip(),
                    'artifact': str(row['artifact']).strip(),
                    'prompt': row['prompt'],
                    'quality': quality_score,
                    'prompt_name': filename_prompt
                }
                all_annotations.append(annotation)
        
        except Exception as e:
            traceback.print_exc()
            continue
    
    if len(all_annotations) == 0:
        raise ValueError("No valid annotations found. Check that prompt names match between quality CSV and tagged artifact filenames.")
    
    
    def weighted_similarity(a, b):
        continent_sim = 1.0 if a[0] == b[0] else 0.0
        country_sim = 1.0 if a[1] == b[1] else 0.0
        artifact_sim = 1.0 if a[2] == b[2] else 0.0
        
        return w1 * continent_sim + w2 * country_sim + w3 * artifact_sim
    
    chunks = [all_annotations[i:i + batch_size] 
              for i in range(0, len(all_annotations), batch_size)]
        
    all_qvs = []
    all_vs = []
    all_quality = []
    
    for chunk in chunks:
        if len(chunk) < 2:  
            continue

        samples = [(item['continent'], item['country'], item['artifact']) 
                   for item in chunk]
        
        qualities = np.array([item['quality'] for item in chunk])
        mean_quality = qualities.mean()
        
        vs = vendi.score(samples, k=weighted_similarity)
        
        normalized_vs = vs / len(chunk)
        

        qvs = mean_quality * normalized_vs
        
        all_qvs.append(qvs)
        all_vs.append(normalized_vs)
        all_quality.append(mean_quality)

    mean_qvs = np.mean(all_qvs)
    mean_vs = np.mean(all_vs)
    mean_quality = np.mean(all_quality)
    
    if return_components:
        return {
            'q': mean_quality,           # Average quality
            'vs': mean_vs,               # Average normalized Vendi score (diversity)
            'qvs': mean_qvs,             # Cultural Diversity (quality-weighted)
            'n_batches': len(all_qvs),
            'n_samples': len(all_annotations),
            'weights': (w1, w2, w3)
        }
    else:
        return mean_qvs


def calculate_all_kernel_configurations(
    quality_scores_csv,
    tagged_artifacts_folder,
    batch_size=8
):
    
    configurations = {
        'Continent-level (1,0,0)': (1.0, 0.0, 0.0),
        'Country-level (0,1,0)': (0.0, 1.0, 0.0),
        'Artifact-level (0,0,1)': (0.0, 0.0, 1.0),
        'Hierarchical (1/2,1/2,0)': (0.5, 0.5, 0.0),
        'Uniform (1/3,1/3,1/3)': (1/3, 1/3, 1/3)
    }
    
    results = {}
    
    concept_name = Path(tagged_artifacts_folder).name
    
    quality_result = calculate_cultural_diversity(
        quality_scores_csv,
        tagged_artifacts_folder,
        w1=1.0, w2=0.0, w3=0.0,
        batch_size=batch_size,
        return_components=True
    )
    
    results['concept'] = concept_name
    results['quality'] = quality_result['q']
    results['n_samples'] = quality_result['n_samples']
    
    for config_name, (w1, w2, w3) in configurations.items():
    
        result = calculate_cultural_diversity(
            quality_scores_csv,
            tagged_artifacts_folder,
            w1=w1, w2=w2, w3=w3,
            batch_size=batch_size,
            return_components=True
        )
        
        results[config_name] = {
            'vs': result['vs'],
            'qvs': result['qvs'],
            'weights': (w1, w2, w3)
        }
        
        print(f"{config_name:<35} {result['vs']:<12.4f} {result['qvs']:<12.4f}")
    

    return results

def save_results_to_csv(results, output_csv):
    rows = []
    
    concept = results['concept']
    quality = results['quality']
    n_samples = results['n_samples']
    
    row = {
        'Concept': concept,
        'Metric': 'Quality (q)',
        'Value': f"{quality:.4f}",
        'N_Samples': n_samples
    }
    rows.append(row)
  
    for config_name in ['Continent-level (1,0,0)', 'Country-level (0,1,0)', 
                        'Artifact-level (0,0,1)', 'Hierarchical (1/2,1/2,0)', 
                        'Uniform (1/3,1/3,1/3)']:
        if config_name in results:
            row = {
                'Concept': concept,
                'Metric': f'VS {config_name}',
                'Value': f"{results[config_name]['vs']:.4f}",
                'N_Samples': n_samples
            }
            rows.append(row)
    
    for config_name in ['Continent-level (1,0,0)', 'Country-level (0,1,0)', 
                        'Artifact-level (0,0,1)', 'Hierarchical (1/2,1/2,0)', 
                        'Uniform (1/3,1/3,1/3)']:
        if config_name in results:
            row = {
                'Concept': concept,
                'Metric': f'qVS (CD) {config_name}',
                'Value': f"{results[config_name]['qvs']:.4f}",
                'N_Samples': n_samples
            }
            rows.append(row)
    
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)

    return df

In [ ]:
weihts_comb = [
    (1, 0, 0),
    (0, 1, 0),
    (0, 0, 1),
    (0.5, 0.5, 0),
    (1/3, 1/3, 1/3)
]

results = []
for w1, w2, w3 in weihts_comb:
    cd_score = calculate_cultural_diversity(
        quality_scores_csv='<LOCAL_QUALITY_CSV_PATH>',
        tagged_artifacts_folder='<LOCAL_TAGGED_ARTIFACTS_PATH>',
        w1=w1,  # Continent weight
        w2=w2,  # Country weight  
        w3=w3,  # Artifact weight
        batch_size=8
    )
    results.append(f"Weights ({w1}, {w2}, {w3}) -> Cultural Diversity (qVS): {cd_score:.4f}")

In [26]:

for i, res in enumerate(results):
    print(f"combination {weihts_comb[i]}: {res}")

combination (1, 0, 0): Weights (1, 0, 0) -> Cultural Diversity (qVS): 0.0508
combination (0, 1, 0): Weights (0, 1, 0) -> Cultural Diversity (qVS): 0.1490
combination (0, 0, 1): Weights (0, 0, 1) -> Cultural Diversity (qVS): 0.2168
combination (0.5, 0.5, 0): Weights (0.5, 0.5, 0) -> Cultural Diversity (qVS): 0.1155
combination (0.3333333333333333, 0.3333333333333333, 0.3333333333333333): Weights (0.3333333333333333, 0.3333333333333333, 0.3333333333333333) -> Cultural Diversity (qVS): 0.1678
